# BioNeMo Inference Runtime (BIR) — Boltz-2 tutorial

This notebook teaches the core BIR workflow: describe a protein request, configure a processor, run it, and inspect the generated structure and scores. It runs in the Nebius Serverless image, where the public BIR package is installed before Jupyter starts.

> BIR is an inference-acceleration **library**, not a NIM or a prebuilt model-serving API. Boltz-2 is one public model example.

## 1. Import BIR's request types and pipeline builder

The image has already bootstrapped the public `bionemo-ir` distribution. To reproduce this outside the image, install the public BIR release appropriate for your CUDA, Python, and platform before running these cells.

In [ ]:
from pathlib import Path

from bionemo_ir.data.schemas import InputRequest, MSARecord, Polymer
from bionemo_ir.pipeline.processor.engine_proc import EngineProcessorConfig, build_processor
from bionemo_ir.pipeline.stages.configs import FeatureGeneratorStageConfig, WriterStageConfig


## 2. Describe one protein input

For a self-contained demonstration, this uses a query-only A3M alignment. Supply a production-quality alignment for scientific work.

In [ ]:
sequence = (
    'ACKIENIKYKGKEVESKLGSQLIDIFNDLDRAKEEYDKLSSPEFIAKFGDWINDEVERNVN'
    'EDGEPLLIQDVRQDSSKHYFFILKNGERFDLLTR'
)
request = InputRequest(
    input_id='boltz2-tutorial',
    polymers=[
        Polymer(
            chain_id=['A1'],
            sequence=sequence,
            msas=[MSARecord(content=f'>query\n{sequence}\n')],
        )
    ],
)
request


## 3. Build a reusable Boltz-2 processor

Processor creation prepares BIR and the model. Keep the processor resident and reuse it for subsequent requests rather than rebuilding it for every sequence. The first run downloads public model assets and can take several minutes.

In [ ]:
output_dir = Path('/workspace/output')
output_dir.mkdir(parents=True, exist_ok=True)

processor = build_processor(
    EngineProcessorConfig(
        model_source='boltz-2',
        runtime_args={'num_sampling_steps': 50},
        feature_generator_stage=FeatureGeneratorStageConfig(
            init_context={'random_seed': 42}
        ),
        writer_stage=WriterStageConfig(output_path=str(output_dir), format='cif'),
        engine_kwargs={'profile_inference': True},
    )
)


## 4. Run inference

BIR processors take records in a list and return one result row per record. Each row carries the generated mmCIF path, scores, and profiling fields.

In [ ]:
rows = processor([{'record': request, '__record_id': request.input_id}])
row = rows[0]
row


## 5. Inspect the structure and scores

`output_path` points to the mmCIF produced by the writer stage. Treat scores and structures as model outputs that need domain-appropriate validation; this notebook is an integration tutorial, not a benchmark or scientific validation protocol.

In [ ]:
import json

cif_path = Path(row['output_path'])
scores = json.loads(row['scores'])
print(f'mmCIF: {cif_path}')
print(f'Inference time: {row.get("model_inference_time", "not reported")}')
print('Score keys:', sorted(scores))
print(cif_path.read_text(encoding='utf-8')[:500])


## Next steps

- Replace the query-only MSA with an alignment appropriate for your target.
- Change `num_sampling_steps`, then measure accuracy, memory, and runtime on your target GPU.
- For an HTTP integration, see the sibling `src/server.py` example in this cookbook template. It keeps one processor resident and maps a request to the same BIR objects used here.
- Pin the public BIR version in the Serverless endpoint environment for reproducible work.